In [6]:
# ============================================================
# INSTALL + GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers datasets accelerate scikit-learn

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# ============================================================
# CENTRALIZED BERT-BASE FULL FINE-TUNING FOR SST-2
# ============================================================

import csv
import json
import logging
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch

from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# ============================================================
# CONFIG
# ============================================================

MODEL_NAME   = "bert-base-uncased"
DATASET_NAME = "glue"
DATASET_CFG  = "sst2"

NUM_LABELS   = 2
TEXT_COLUMN  = "sentence"
LABEL_COLUMN = "label"

SETTING_TAG  = "C-BERT-base-FFT"

BATCH_SIZE   = 32
LEARNING_RATE = 2e-5
EPOCHS       = 30
MAX_LENGTH   = 128

GRAD_ACCUM   = 1
WARMUP_RATIO = 0.06
PATIENCE     = 3
SEED         = 42

OUTPUT_DIR = "/content/drive/MyDrive/cen_bert_sst2"

# ============================================================
# LOGGER
# ============================================================

def setup_logger(log_path: Path):

    log_path.parent.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger(SETTING_TAG)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter(
        "[%(asctime)s] %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setFormatter(fmt)

    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(sh)

    return logger

# ============================================================
# SEED
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================
# DATASET
# ============================================================

def load_and_tokenize(tokenizer, max_length):

    ds = load_dataset(DATASET_NAME, DATASET_CFG)

    train_ds = ds["train"]
    eval_ds  = ds["validation"]

    def tok_fn(batch):

        return tokenizer(
            batch[TEXT_COLUMN],
            truncation=True,
            max_length=max_length
        )

    train_ds = train_ds.map(
        tok_fn,
        batched=True,
        remove_columns=[TEXT_COLUMN]
    )

    eval_ds = eval_ds.map(
        tok_fn,
        batched=True,
        remove_columns=[TEXT_COLUMN]
    )

    for c in list(train_ds.column_names):

        if c not in ("input_ids", "attention_mask", LABEL_COLUMN):
            train_ds = train_ds.remove_columns([c])

    for c in list(eval_ds.column_names):

        if c not in ("input_ids", "attention_mask", LABEL_COLUMN):
            eval_ds = eval_ds.remove_columns([c])

    train_ds = train_ds.rename_column(LABEL_COLUMN, "labels")
    eval_ds  = eval_ds.rename_column(LABEL_COLUMN, "labels")

    train_ds.set_format("torch")
    eval_ds.set_format("torch")

    return train_ds, eval_ds

# ============================================================
# MODEL
# ============================================================

def build_model():

    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS
    )

# ============================================================
# PARAM COUNT
# ============================================================

def count_params(model):

    total = sum(p.numel() for p in model.parameters())

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return trainable, total

# ============================================================
# TRAIN
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    scaler,
    device,
):

    model.train()

    losses = []

    t0 = time.time()

    optimizer.zero_grad(set_to_none=True)

    step = 0

    for batch in loader:

        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        with autocast(dtype=torch.float16):

            outputs = model(**batch)

            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()

        losses.append(loss.item() * GRAD_ACCUM)

        step += 1

        if step % GRAD_ACCUM == 0:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            optimizer.zero_grad(set_to_none=True)

    avg_loss = float(np.mean(losses))

    epoch_time = time.time() - t0

    return avg_loss, epoch_time

# ============================================================
# EVALUATE
# ============================================================

@torch.no_grad()
def evaluate(model, loader, device):

    model.eval()

    losses = []

    preds = []
    golds = []

    for batch in loader:

        batch = {
            k: v.to(device, non_blocking=True)
            for k, v in batch.items()
        }

        with autocast(dtype=torch.float16):

            outputs = model(**batch)

        losses.append(outputs.loss.item())

        pred = outputs.logits.argmax(-1)

        preds.extend(pred.cpu().tolist())

        golds.extend(batch["labels"].cpu().tolist())

    metrics = {

        "eval_loss":
            float(np.mean(losses)),

        "accuracy":
            accuracy_score(golds, preds),

        "precision":
            precision_score(
                golds,
                preds,
                average="macro",
                zero_division=0
            ),

        "recall":
            recall_score(
                golds,
                preds,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                golds,
                preds,
                average="macro",
                zero_division=0
            ),
    }

    return metrics

# ============================================================
# CHECKPOINT
# ============================================================

class CheckpointManager:

    def __init__(self, directory, max_keep=2):

        self.directory = directory
        self.max_keep  = max_keep

        directory.mkdir(parents=True, exist_ok=True)

    def save(self, payload, epoch):

        path = self.directory / f"checkpoint_epoch_{epoch:04d}.pt"

        torch.save(payload, path)

        self.prune()

        return path

    def prune(self):

        ckpts = sorted(
            self.directory.glob("checkpoint_epoch_*.pt")
        )

        while len(ckpts) > self.max_keep:

            try:
                ckpts.pop(0).unlink()
            except:
                pass

    def latest(self):

        ckpts = sorted(
            self.directory.glob("checkpoint_epoch_*.pt")
        )

        if len(ckpts) == 0:
            return None

        return ckpts[-1]

# ============================================================
# CSV
# ============================================================

CSV_FIELDS = [

    "epoch",

    "train_loss",
    "eval_loss",

    "accuracy",
    "precision",
    "recall",
    "macro_f1",

    "time_per_epoch",

    "trainable_params",
    "total_params",

    "model_name",
    "dataset_name",
    "setting",

    "best_metric_so_far",

    "patience_counter",

    "is_new_best",
]

def append_history(csv_path, json_path, row, history):

    write_header = not csv_path.exists()

    with open(csv_path, "a", newline="", encoding="utf-8") as f:

        writer = csv.DictWriter(
            f,
            fieldnames=CSV_FIELDS
        )

        if write_header:
            writer.writeheader()

        writer.writerow({
            k: row.get(k, "")
            for k in CSV_FIELDS
        })

    history.append(row)

    with open(json_path, "w", encoding="utf-8") as f:

        json.dump(history, f, indent=2)

# ============================================================
# MAIN
# ============================================================

def main():

    set_seed(SEED)

    out = Path(OUTPUT_DIR)

    out.mkdir(parents=True, exist_ok=True)

    ckpt_dir = out / "checkpoints"
    best_dir = out / "best_model"
    final_dir = out / "final_model"

    logger = setup_logger(out / "train.log")

    history_csv  = out / "history.csv"
    history_json = out / "history.json"

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

    print("=" * 80)
    print(f"MODEL_NAME : {MODEL_NAME}")
    print(f"SETTING    : {SETTING_TAG}")
    print(f"DEVICE     : {device}")
    print("=" * 80)

    if torch.cuda.is_available():

        print("RUNNING ON GPU")
        print(f"GPU NAME        : {torch.cuda.get_device_name(0)}")

        total_vram = (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        )

        print(f"TOTAL VRAM      : {total_vram:.2f} GB")

        print(f"CUDA VERSION    : {torch.version.cuda}")

    else:
        print("RUNNING ON CPU")

    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    train_ds, eval_ds = load_and_tokenize(
        tokenizer,
        MAX_LENGTH
    )

    collator = DataCollatorWithPadding(tokenizer)

    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True,
    )

    eval_loader = DataLoader(
        eval_ds,
        batch_size=BATCH_SIZE * 2,
        shuffle=False,
        collate_fn=collator,
        num_workers=2,
        pin_memory=True,
    )

    model = build_model().to(device)

    trainable, total = count_params(model)

    logger.info(
        f"trainable={trainable:,} total={total:,}"
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )

    steps_per_epoch = max(
        1,
        len(train_loader) // GRAD_ACCUM
    )

    total_steps = steps_per_epoch * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )

    scaler = GradScaler()

    ckpt_mgr = CheckpointManager(ckpt_dir)

    history = []

    start_epoch = 1

    best_metric = -float("inf")

    patience_counter = 0

    latest = ckpt_mgr.latest()

    # ============================================================
    # RESUME
    # ============================================================

    if latest is not None:

        logger.info(f"Resuming from {latest}")

        ckpt = torch.load(
            latest,
            map_location="cpu"
        )

        model.load_state_dict(ckpt["model"])

        optimizer.load_state_dict(ckpt["optimizer"])

        scheduler.load_state_dict(ckpt["scheduler"])

        scaler.load_state_dict(ckpt["scaler"])

        start_epoch = ckpt["epoch"] + 1

        best_metric = ckpt.get(
            "best_metric",
            -float("inf")
        )

        patience_counter = ckpt.get(
            "patience_counter",
            0
        )

        history = ckpt.get("history", [])

    # ============================================================
    # TRAIN LOOP
    # ============================================================

    for epoch in range(start_epoch, EPOCHS + 1):

        logger.info(f"==== Epoch {epoch}/{EPOCHS} ====")

        train_loss, epoch_time = train_one_epoch(
            model,
            train_loader,
            optimizer,
            scheduler,
            scaler,
            device,
        )

        metrics = evaluate(
            model,
            eval_loader,
            device,
        )

        is_new_best = metrics["accuracy"] > best_metric

        if is_new_best:

            best_metric = metrics["accuracy"]

            patience_counter = 0

            best_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            model.save_pretrained(best_dir)

            tokenizer.save_pretrained(best_dir)

        else:

            patience_counter += 1

        row = {

            "epoch": epoch,

            "train_loss":
                train_loss,

            "eval_loss":
                metrics["eval_loss"],

            "accuracy":
                metrics["accuracy"],

            "precision":
                metrics["precision"],

            "recall":
                metrics["recall"],

            "macro_f1":
                metrics["macro_f1"],

            "time_per_epoch":
                epoch_time,

            "trainable_params":
                trainable,

            "total_params":
                total,

            "model_name":
                MODEL_NAME,

            "dataset_name":
                "glue/sst2",

            "setting":
                SETTING_TAG,

            "best_metric_so_far":
                best_metric,

            "patience_counter":
                patience_counter,

            "is_new_best":
                int(is_new_best),
        }

        append_history(
            history_csv,
            history_json,
            row,
            history,
        )

        logger.info(
            f"loss={train_loss:.4f} | "
            f"acc={metrics['accuracy']:.4f} | "
            f"f1={metrics['macro_f1']:.4f} | "
            f"best={best_metric:.4f} | "
            f"patience={patience_counter} | "
            f"time={epoch_time:.1f}s"
        )

        ckpt_mgr.save({

            "epoch": epoch,

            "model": model.state_dict(),

            "optimizer": optimizer.state_dict(),

            "scheduler": scheduler.state_dict(),

            "scaler": scaler.state_dict(),

            "best_metric": best_metric,

            "patience_counter": patience_counter,

            "history": history,

        }, epoch)

        if patience_counter >= PATIENCE:

            logger.info(
                f"Early stopping at epoch {epoch}"
            )

            break

    # ============================================================
    # SAVE FINAL MODEL
    # ============================================================

    final_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    model.save_pretrained(final_dir)

    tokenizer.save_pretrained(final_dir)

    logger.info(
        f"Done. Best accuracy={best_metric:.4f}"
    )

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":

    main()

MODEL_NAME : bert-base-uncased
SETTING    : C-BERT-base-FFT
DEVICE     : cuda
RUNNING ON GPU
GPU NAME        : Tesla T4
TOTAL VRAM      : 14.56 GB
CUDA VERSION    : 12.8


Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[2026-05-17 09:37:42] INFO | Early stopping at epoch 8


INFO:C-BERT-base-FFT:Early stopping at epoch 8


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[2026-05-17 09:37:45] INFO | Done. Best accuracy=0.9243


INFO:C-BERT-base-FFT:Done. Best accuracy=0.9243
